
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>


# Reading from a Streaming Query

##### Objectives
1. Build streaming DataFrames
1. Display streaming query results
1. Write streaming query results
1. Monitor a streaming query

##### Classes
- <a href="https://spark.apache.org/docs/latest/api/python/reference/pyspark.ss/api/pyspark.sql.streaming.DataStreamReader.html" target="_blank">DataStreamReader</a>
- <a href="https://spark.apache.org/docs/latest/api/python/reference/pyspark.ss/api/pyspark.sql.streaming.DataStreamWriter.html" target="_blank">DataStreamWriter</a>
- <a href="https://spark.apache.org/docs/latest/api/python/reference/pyspark.ss/api/pyspark.sql.streaming.StreamingQuery.html" target="_blank">StreamingQuery</a>

In [0]:
%run ./Includes/Classroom-Setup-01.1


### Build streaming DataFrames

Obtain an initial streaming DataFrame from a Delta-format file source.

In [0]:
df = (spark
      .readStream
      .format("delta")
      .load(DA.paths.events)
)

print("Is the dataframe streaming:", df.isStreaming)
df.printSchema()

Apply some transformations, producing new streaming DataFrames.

In [0]:
from pyspark.sql.functions import col, approx_count_distinct, count
# import F. Also, cell imports in 
email_traffic_df = (df
                    .filter(col("traffic_source") == "email")
                    .withColumn("mobile", col("device").isin(["iOS", "Android"]))
                    .select("user_id", "event_timestamp", "mobile")
                   )

email_traffic_df.isStreaming


### Write streaming query results

Take the final streaming DataFrame (our result table) and write it to a file sink in "append" mode.

In [0]:
checkpoint_path = f"{DA.paths.working_dir}/email_traffic"
output_path = f"{DA.paths.working_dir}/email_traffic/output"

devices_query = (email_traffic_df
                 .writeStream
                 .outputMode("append")
                 .format("delta") # Although default is Delta, we're explicitly calling this out. This line is vestigial as of DBR 8.0
                 .queryName("email_traffic")
                 .trigger(processingTime="1 second")
                 .option("checkpointLocation", checkpoint_path)
                 .start(output_path)
                )

In [0]:
DA.block_until_stream_is_ready(devices_query)

### Monitor streaming query

Use the streaming query handle to monitor and control it.

In [0]:
devices_query.id


Query status output

In [0]:
devices_query.status

[lastProgress](https://spark.apache.org/docs/3.1.2/api/python/reference/api/pyspark.sql.streaming.StreamingQuery.lastProgress.html) gives us metrics from the previous query

In [0]:
devices_query.lastProgress

In [0]:
import time
# Run for 10 more seconds
time.sleep(10) 

devices_query.stop()

In [0]:
devices_query.awaitTermination()

[awaitTermination](https://spark.apache.org/docs/3.1.1/api/python/reference/api/pyspark.sql.streaming.StreamingQuery.awaitTermination.html) blocks the current thread until the streaming query is terminated. In our course notebooks we use it in case you use "Run All" to run the notebook to prevent subsequent command cells from executing until the streaming query has fully terminated.

For stand-alone structured streaming applications, this is used to prevent the main thread from terminating while the streaming query is still executing.


### Classroom Cleanup
Run the cell below to clean up resources.

In [0]:
DA.cleanup()


&copy; 2024 Databricks, Inc. All rights reserved.<br/>
Apache, Apache Spark, Spark and the Spark logo are trademarks of the 
<a href="https://www.apache.org/">Apache Software Foundation</a>.<br/>
<br/><a href="https://databricks.com/privacy-policy">Privacy Policy</a> | 
<a href="https://databricks.com/terms-of-use">Terms of Use</a> | 
<a href="https://help.databricks.com/">Support</a>